# Energy budgets

Compute a time series for the energy components and exchanges

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
from scores.budgets import energy_components, energy_exchanges
import matplotlib.pyplot as plt

In [ ]:
START_TIME = pd.Timestamp("2020-01-01")
END_TIME = pd.Timestamp("2020-02-01")
ds = xr.open_zarr("gs://weatherbench2/datasets/era5/1959-2023_01_10-6h-240x121_equiangular_with_poles_conservative.zarr")
ds = ds.sel(time=slice(START_TIME, END_TIME))

Prepare the fields for the energy budget time series, using the zonal, meridional and vertical velocities, the temperature, water vapor, surface pressure and surface geopotential (the orography). Also re-order the dimensions are required.

In [ ]:
fieldnames = ['u_component_of_wind','v_component_of_wind','vertical_velocity','specific_humidity','temperature','geopotential','surface_pressure','geopotential_at_surface']
ds = ds[fieldnames]
ds['u_component_of_wind'] = ds['u_component_of_wind'].transpose('time','level','latitude','longitude')
ds['v_component_of_wind'] = ds['v_component_of_wind'].transpose('time','level','latitude','longitude')
ds['vertical_velocity'] = ds['vertical_velocity'].transpose('time','level','latitude','longitude')
ds['specific_humidity'] = ds['specific_humidity'].transpose('time','level','latitude','longitude')
ds['temperature'] = ds['temperature'].transpose('time','level','latitude','longitude')
ds['geopotential'] = ds['geopotential'].transpose('time','level','latitude','longitude')
ds['surface_pressure'] = ds['surface_pressure'].transpose('time','latitude','longitude')
ds['geopotential_at_surface'] = ds['geopotential_at_surface'].transpose('latitude','longitude')

Specify a sub-domain of the global to compute the budgets over

In [ ]:
sub_domain_longitude = np.array([None,None])
sub_domain_latitude = np.array([-60.0,-20.0])

Compute a time series for the internal, latent, potential and horizontal and vertical kinetic energies:

Internal = $\int_{p_1}^{p_0}\int_{\Omega}(C_p^d(1-q) + C_p^vq)T\mathrm{d}\Omega\mathrm{d}p$ 

Latent = $\int_{p_1}^{p_0}\int_{\Omega}L_vq\mathrm{d}\Omega\mathrm{d}p$ 

Potential = $\int_{\Omega}z_s\Phi_s\mathrm{d}\Omega$ 

Kinetic (horizontal) = $\int_{p_1}^{p_0}\int_{\Omega}\frac{1}{2}(u^2 + v^2)\mathrm{d}\Omega\mathrm{d}p$ 

Kinetic (vertical) = $\int_{p_1}^{p_0}\int_{\Omega}\frac{1}{2}w^2\mathrm{d}\Omega\mathrm{d}p$ 

Reference: Trenberth et. al. J. Clim. (2002) v. 15 pp 3343--3360, eqn (5) Sha et. al (2025), eqn (12)

In [ ]:
E = energy_components(ds, fieldnames, sub_domain_longitude, sub_domain_latitude, 'budget_' + START_TIME.strftime('%Y-%m-%d-%H:%M:%S') + '.txt')

In [ ]:
time=0.25*np.arange(E.shape[0])
e_names = ['Internal','Latent','Potential','Kinetic (horiz.)','Kinetic (vert.)']
plt.semilogy(time,E[:,0])
plt.semilogy(time,E[:,1])
plt.semilogy(time,E[:,2])
plt.semilogy(time,E[:,3])
plt.semilogy(time,E[:,4])
plt.legend(e_names)
plt.xlabel('time (days)')
plt.show()

In [ ]:
fields = ['u','v','z']
files = []
for field in fields:
    files.append(input_path + field + '_' + date + '_6h_1008_em1.nc')

ds = xr.open_mfdataset(files,combine='by_coords')
ds = ds.sortby("latitude")

ds_zs = xr.open_dataset("/g/data/dx2/ML/output_data/hindcast/AIFS/outputs/postproc/ens/ensv1/ifs_mars_20240101T000000_360.nc")
ds_zs = ds_zs["z_surf"].sel(time=ds_zs.time[0])
ds["z_surf"] = ds_zs

Compute a time series for the kinetic to internal, internal to kinetic, kinetic to potential and potential to kinetic energy exchanges as:

Kinetic to Internal = $\int_{p_1}^{p_0}\int_{\Omega}\nabla(z-z_s)\cdot\boldsymbol{u}\mathrm{d}\Omega\mathrm{d}p$

Internal to Kinetic = $\int_{p_1}^{p_0}\int_{\Omega}(z-z_s)\nabla\cdot\boldsymbol{u}\mathrm{d}\Omega\mathrm{d}p$

Kinetic to Potential = $\int_{p_1}^{p_0}\int_{\Omega}\nabla z_s\cdot\boldsymbol{u}\mathrm{d}\Omega\mathrm{d}p$

Potential to Kinetic = $\int_{p_1}^{p_0}\int_{\Omega}z_s\nabla\cdot\boldsymbol{u}\mathrm{d}\Omega\mathrm{d}p$

In [ ]:
Ex = energy_exchanges(ds, fieldnames, sub_domain_longitude, sub_domain_latitude, 'exchanges_' + START_TIME.strftime('%Y-%m-%d-%H:%M:%S') + '.txt')

In [ ]:
time=0.25*np.arange(Ex.shape[0])
ex_names = ['Kinetic to Internal','Internal to Kinetic','Kinetic to Potential','Potential to Kinetic']
plt.plot(time,Ex[:,0])
plt.plot(time,Ex[:,1])
plt.plot(time,Ex[:,2])
plt.plot(time,Ex[:,3])
plt.legend(ex_names)
plt.xlabel('time (days)')
plt.show()